<a href="https://colab.research.google.com/github/wlau818/Ocular-Anomaly/blob/main/Ocular_Anomaly_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import os
import shutil
import pandas as pd

In [2]:
# 1. Define your local project target paths
PROJECT_DATA_DIR = "data"
LOCAL_CSV_PATH = os.path.join(PROJECT_DATA_DIR, "full_df.csv")
LOCAL_CSV_PATH_2 = os.path.join(PROJECT_DATA_DIR, "metadata.csv")


# 2. Create the 'data' folder inside your project if it doesn't exist
if not os.path.exists(PROJECT_DATA_DIR):
    os.makedirs(PROJECT_DATA_DIR)
    print(f"Created local directory: {PROJECT_DATA_DIR}")

# 3. Only download and copy if the file isn't already in your project
if not os.path.exists(LOCAL_CSV_PATH):
    print("Fetching file from cache...")
    cache_dir = kagglehub.dataset_download("andrewmvd/ocular-disease-recognition-odir5k")
    cache_csv_path = os.path.join(cache_dir, "full_df.csv")

    # Copy the file to your project folder
    shutil.copy(cache_csv_path, LOCAL_CSV_PATH)
    print(f"Successfully copied full_df.csv to local path: {LOCAL_CSV_PATH}")
else:
    print(f"File already exists locally at: {LOCAL_CSV_PATH}")

# creating the second dataset file
if not os.path.exists(LOCAL_CSV_PATH_2):
    print("Fetching file from cache...")
    cache_dir = kagglehub.dataset_download("deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2")

    # -> THE FIX: Added the specific nested folder where Kaggle hid the metadata file <-
    cache_csv_path = os.path.join(cache_dir, "eyepac-light-v2-512-jpg", "metadata.csv")

    # Copy the file to your project folder
    shutil.copy(cache_csv_path, LOCAL_CSV_PATH_2)
    print(f"Successfully copied metadata.csv to local path: {LOCAL_CSV_PATH_2}")
else:
    print(f"File already exists locally at: {LOCAL_CSV_PATH_2}")

# 4. Clean load using your new local project path
df = pd.read_csv(
    LOCAL_CSV_PATH,
    encoding="utf-8",
    encoding_errors="replace",
    engine="python",
    on_bad_lines="skip"
)

df_2 = pd.read_csv(
    LOCAL_CSV_PATH_2,
    encoding="utf-8",
    encoding_errors="replace",
    engine="python",
    on_bad_lines="skip"
)

print(f"\nLoaded {len(df)} rows from your project folder.")
print(f"\nLoaded {len(df_2)} rows from your project folder.")

Created local directory: data
Fetching file from cache...
Using Colab cache for faster access to the 'ocular-disease-recognition-odir5k' dataset.
Successfully copied full_df.csv to local path: data/full_df.csv
Fetching file from cache...


100%|██████████| 524M/524M [00:06<00:00, 84.2MB/s]

Extracting files...


Successfully copied metadata.csv to local path: data/metadata.csv

Loaded 6392 rows from your project folder.

Loaded 9540 rows from your project folder.


In [3]:
# Dataset 1
print(df.shape)
df.head(5)


(6392, 19)


,ID,Patient Age,Patient Sex,Left-Fundus,Right-Fundus,Left-Diagnostic Keywords,Right-Diagnostic Keywords,N,D,G,C,A,H,M,O,filepath,labels,target,filename
0,0,69,Female,0_left.jpg,0_right.jpg,cataract,normal fundus,0,0,0,1,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",0_right.jpg
1,1,57,Male,1_left.jpg,1_right.jpg,normal fundus,normal fundus,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",1_right.jpg
2,2,42,Male,2_left.jpg,2_right.jpg,laser spot，moderate non proliferative retinopathy,moderate non proliferative retinopathy,0,1,0,0,0,0,0,1,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",2_right.jpg
3,4,53,Male,4_left.jpg,4_right.jpg,macular epiretinal membrane,mild nonproliferative retinopathy,0,1,0,0,0,0,0,1,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",4_right.jpg
4,5,50,Female,5_left.jpg,5_right.jpg,moderate non proliferative retinopathy,moderate non proliferative retinopathy,0,1,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",5_right.jpg


In [4]:
df.isna().sum().sum()
(df['ID'].value_counts() == 1).sum()

np.int64(324)

In [5]:
## Checking the second dataset

print(df_2.shape)
print(df_2.head(5))

(9540, 8)
     id                  file_name label  label_binary      folder  \
0  2580  EyePACS-TRAIN-RG-2580.jpg    RG             1  validation   
1  2617  EyePACS-TRAIN-RG-2617.jpg    RG             1  validation   
2  2794  EyePACS-TRAIN-RG-2794.jpg    RG             1  validation   
3  2696  EyePACS-TRAIN-RG-2696.jpg    RG             1  validation   
4  2585  EyePACS-TRAIN-RG-2585.jpg    RG             1  validation   

  source_dataset relative_file_type                                file_path  
0  EyePACS-TRAIN                jpg  /eyepac-light-v2-512-jpg/validation/RG/  
1  EyePACS-TRAIN                jpg  /eyepac-light-v2-512-jpg/validation/RG/  
2  EyePACS-TRAIN                jpg  /eyepac-light-v2-512-jpg/validation/RG/  
3  EyePACS-TRAIN                jpg  /eyepac-light-v2-512-jpg/validation/RG/  
4  EyePACS-TRAIN                jpg  /eyepac-light-v2-512-jpg/validation/RG/  


## Checking for duplicate rows in Dataset 1

There are duplicate rows for one person, difference is the filepath and filename, each for right and left eye for a given person.

This leads to duplicated rows when excluding the filepath and filename in the new `df` where we turn each left and right eye into their own row.   

In [6]:
df.loc[df['Left-Fundus'] == '1_left.jpg']

,ID,Patient Age,Patient Sex,Left-Fundus,Right-Fundus,Left-Diagnostic Keywords,Right-Diagnostic Keywords,N,D,G,C,A,H,M,O,filepath,labels,target,filename
1,1,57,Male,1_left.jpg,1_right.jpg,normal fundus,normal fundus,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",1_right.jpg
3195,1,57,Male,1_left.jpg,1_right.jpg,normal fundus,normal fundus,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",1_left.jpg


In [7]:
rows = df.loc[df['Left-Fundus'] == '1_left.jpg', 'filepath']

for p in rows:
    print(p)

../input/ocular-disease-recognition-odir5k/ODIR-5K/Training Images/1_right.jpg
../input/ocular-disease-recognition-odir5k/ODIR-5K/Training Images/1_left.jpg


In [8]:
(df['Left-Fundus'].value_counts() == 1).sum()

np.int64(324)

## Filtering for the Glaucoma and Normal rows in Dataset 1

1. Break up each rows to separate between left and right eye (SQL)
2. Filter out the rows specifically for glaucoma and normal
3. Combine the glaucoma and normal dataset into one df

In [9]:
import sqlite3

def split_row_data(df, check_glaucoma = False):

    conn = sqlite3.connect(":memory:")
    df.to_sql("df", conn, index=False, if_exists="replace")

    # Please check, using DISTINCT to get rid of the repetitive rows where the only differences were the filepaths
    query = """
    SELECT DISTINCT
        ID,
        "Patient Age",
        "Patient Sex",
        "Left-Fundus" AS file_name,
        "Left-Diagnostic Keywords" AS label,
        G as label_binary,
        'left' AS eye_side,
        'ODIR-5K' AS source_dataset
    FROM
        df

    UNION ALL

    SELECT DISTINCT
        ID,
        "Patient Age",
        "Patient Sex",
        "Right-Fundus" AS file_name,
        "Right-Diagnostic Keywords" AS label,
        G as label_binary,
        'right' AS eye_side,
        'ODIR-5K' AS source_dataset
    FROM
        df
    ;
    """
    # apply the query to get a df of the split rows
    out = pd.read_sql_query(query, conn)

    # check for glaucoma if true
    if check_glaucoma == True:
        out = out[out["label"].str.contains("glaucoma", na=False)]

    return out


In [10]:
# df_glaucoma = df[df['G'] == 1]
# print(df_glaucoma.shape)
# print(df_glaucoma['G'].unique())

# # df_glaucoma.head()

# (df_glaucoma['ID'].value_counts() == 1).sum()


In [11]:
print(df.shape)

(6392, 19)


In [12]:
df_glaucoma_split = split_row_data(df, check_glaucoma = True)
print(df_glaucoma_split.shape)
print(df_glaucoma_split.head())

print(df_glaucoma_split['label_binary'].unique())

# df_glaucoma_split[df_glaucoma_split["label"].str.contains("glaucoma", na=False)]
# df_glaucoma_split[df_glaucoma_split["label_binary"] == 1]

(316, 8)
      ID  Patient Age Patient Sex     file_name  \
78    95           46        Male   95_left.jpg   
129  153           79        Male  153_left.jpg   
141  167           71        Male  167_left.jpg   
150  178           54        Male  178_left.jpg   
212  247           49        Male  247_left.jpg   

                                             label  label_binary eye_side  \
78                              suspected glaucoma             1     left   
129                                       glaucoma             1     left   
141                                       glaucoma             1     left   
150  dry age-related macular degeneration，glaucoma             1     left   
212  dry age-related macular degeneration，glaucoma             1     left   

    source_dataset  
78         ODIR-5K  
129        ODIR-5K  
141        ODIR-5K  
150        ODIR-5K  
212        ODIR-5K  
[1]


In [13]:
# Checking for duplicated
cols = ["file_name", "label", "label_binary"]

print(df_glaucoma_split[cols].duplicated().sum())

# df_glaucoma_split['file_name'].value_counts()
# print(df_glaucoma_split.loc[df_glaucoma_split['file_name'] == '95_right.jpg'])

# 300 dupes before adding distinct keyword

0


In [14]:
# filtering for normal cases
df_normal = df[df['N'] == 1]
print(df_normal.shape)
print(df_normal.head())

df_normal['G'].unique()

(2101, 19)
      ID  Patient Age Patient Sex   Left-Fundus   Right-Fundus  \
1      1           57        Male    1_left.jpg    1_right.jpg   
7      8           59        Male    8_left.jpg    8_right.jpg   
68    84           51      Female   84_left.jpg   84_right.jpg   
163  191           51      Female  191_left.jpg  191_right.jpg   
344  394           63        Male  394_left.jpg  394_right.jpg   

    Left-Diagnostic Keywords Right-Diagnostic Keywords  N  D  G  C  A  H  M  \
1              normal fundus             normal fundus  1  0  0  0  0  0  0   
7              normal fundus             normal fundus  1  0  0  0  0  0  0   
68             normal fundus             normal fundus  1  0  0  0  0  0  0   
163            normal fundus             normal fundus  1  0  0  0  0  0  0   
344            normal fundus             normal fundus  1  0  0  0  0  0  0   

     O                                           filepath labels  \
1    0  ../input/ocular-disease-recognition-odir5

array([0])

In [15]:
df_normal_split = split_row_data(df_normal, check_glaucoma = False)
print(df_normal_split.shape)
print(df_normal_split.head())


(2160, 8)
    ID  Patient Age Patient Sex     file_name          label  label_binary  \
0    1           57        Male    1_left.jpg  normal fundus             0   
1    8           59        Male    8_left.jpg  normal fundus             0   
2   84           51      Female   84_left.jpg  normal fundus             0   
3  191           51      Female  191_left.jpg  normal fundus             0   
4  394           63        Male  394_left.jpg  normal fundus             0   

  eye_side source_dataset  
0     left        ODIR-5K  
1     left        ODIR-5K  
2     left        ODIR-5K  
3     left        ODIR-5K  
4     left        ODIR-5K  


In [16]:

# Do we want to filter out the low image quality?
print(df_normal_split['label'].unique())
print(df_normal_split.loc[df_normal_split['label'] == 'low image quality'])


['normal fundus' 'normal fundus，lens dust' 'lens dust，normal fundus'
 'normal fundus，normal fundus' 'low image quality']
        ID  Patient Age Patient Sex      file_name              label  \
1048  4290           51        Male  4290_left.jpg  low image quality   

      label_binary eye_side source_dataset  
1048             0     left        ODIR-5K  


In [17]:
cols = ["file_name", "label", "label_binary"]

print(df_normal_split[cols].duplicated().sum())

df_normal_split['file_name'].value_counts()


0


,count
file_name,
3398_right.jpg,1
1_left.jpg,1
8_left.jpg,1
84_left.jpg,1
191_left.jpg,1
...,...
2333_left.jpg,1
2334_left.jpg,1
2335_left.jpg,1


In [18]:
# binary labels using the glaucoma column, called label_binary
    # 0 - normal
    # 1 - glaucoma

filtered_df = pd.concat([df_normal_split, df_glaucoma_split])

print(filtered_df.shape)


(2476, 8)


## Combining the two datasets

We want to keep the columns:
- image/file name
- label (binary 0/1)
- label (string description)

In [19]:
print(filtered_df.head(2))
print(df_2.head(2))

   ID  Patient Age Patient Sex   file_name          label  label_binary  \
0   1           57        Male  1_left.jpg  normal fundus             0   
1   8           59        Male  8_left.jpg  normal fundus             0   

  eye_side source_dataset  
0     left        ODIR-5K  
1     left        ODIR-5K  
     id                  file_name label  label_binary      folder  \
0  2580  EyePACS-TRAIN-RG-2580.jpg    RG             1  validation   
1  2617  EyePACS-TRAIN-RG-2617.jpg    RG             1  validation   

  source_dataset relative_file_type                                file_path  
0  EyePACS-TRAIN                jpg  /eyepac-light-v2-512-jpg/validation/RG/  
1  EyePACS-TRAIN                jpg  /eyepac-light-v2-512-jpg/validation/RG/  


In [20]:
print(filtered_df.columns)      # want to put left fundus and right fundus onto their own row
print(filtered_df.shape)
print(df_2.columns)
print(df_2.shape)

print('Total expected combined rows: ', filtered_df.shape[0] + df_2.shape[0])

Index(['ID', 'Patient Age', 'Patient Sex', 'file_name', 'label',
       'label_binary', 'eye_side', 'source_dataset'],
      dtype='object')
(2476, 8)
Index(['id', 'file_name', 'label', 'label_binary', 'folder', 'source_dataset',
       'relative_file_type', 'file_path'],
      dtype='object')
(9540, 8)
Total expected combined rows:  12016


In [21]:
# Making sure there are no dupliocate rows
cols = ["file_name", "label", "label_binary", "source_dataset"]

print(filtered_df[cols].duplicated().sum())     # have 2342 DUPLICATE ROWS BEFORE SELECT DISTINCT
print(df_2[cols].duplicated().sum())

0
0


In [22]:
conn = sqlite3.connect(":memory:")
filtered_df.to_sql("filtered_df", conn, index=False, if_exists="replace")
df_2.to_sql("df_2", conn, index=False, if_exists="replace")

query = """
SELECT
    file_name,
    label,
    label_binary,
    source_dataset
FROM
    filtered_df

UNION ALL

SELECT
    file_name,
    label,
    label_binary,
    source_dataset
FROM
    df_2
;
"""
# apply the query to get a df of the split rows
combined_df = pd.read_sql_query(query, conn)

print(combined_df.shape)

(12016, 4)


## Train-test split

Splitting our dataset
* 70 train
* 20 test
* 10 validation


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(combined_df, combined_df['label_binary'], test_size=0.3, random_state=123)

X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.66, random_state=123)

In [24]:
print('Training shape: ', X_train.shape)
print('Validation shape: ', X_val.shape)
print('Test shape: ', X_test.shape)

print('\n')
print('Label Distribution by Percentage')
print('Training: \n', y_train.value_counts() / y_train.shape[0])
print('Validation: \n', y_val.value_counts() / y_val.shape[0])
print('Test: \n', y_test.value_counts() / y_test.shape[0])

Training shape:  (8411, 4)
Validation shape:  (1225, 4)
Test shape:  (2380, 4)


Label Distribution by Percentage
Training: 
 label_binary
0    0.576983
1    0.423017
Name: count, dtype: float64
Validation: 
 label_binary
0    0.586939
1    0.413061
Name: count, dtype: float64
Test: 
 label_binary
0    0.570588
1    0.429412
Name: count, dtype: float64


In [25]:
X_train.head()

,file_name,label,label_binary,source_dataset
9369,EyePACS-TRAIN-NRG-2189.jpg,NRG,0,EyePACS-TRAIN
8516,EyePACS-DEV-NRG-1335.jpg,NRG,0,EyePACS-DEV
4329,EyePACS-TRAIN-RG-52.jpg,RG,1,EyePACS-TRAIN
3564,EyePACS-TRAIN-RG-3067.jpg,RG,1,EyePACS-TRAIN
7513,EyePACS-TRAIN-RG-1669.jpg,RG,1,EyePACS-TRAIN


In [26]:
# Checking dataset distribution
X_train['source_dataset'].value_counts()

,count
source_dataset,
EyePACS-TRAIN,4565
EyePACS-DEV,2114
ODIR-5K,1732


##CNN

In [ ]:
# Ensure system headers are installed
!apt-get update && apt-get install -y libopencv-dev

# Run cmake specifically for Linux
!cmake -S . -B build -DPYTHON_EXECUTABLE=$(which python3)

!cmake --build build

!find build -name "*.so" -exec cp {} . \;

In [28]:
# Install system-level build tools
!apt-get update
!apt-get install -y cmake g++ libopencv-dev

# Install pybind11 globally so CMake can find it
!pip install pybind11

# 1. Clean up
!rm -rf build/

# 2. Configure with the explicit path to pybind11
# This command tells CMake where to look for the missing config file
!cmake -S . -B build -Dpybind11_DIR=$(python3 -c "import pybind11;print(pybind11.get_cmake_dir())")

# 3. Build the project
!cmake --build build

# 4. Copy the binary
!find build -name "*.so" -exec cp {} . \;

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [29]:
!ls -F

build/		ocular_cpp.cpython-312-x86_64-linux-gnu.so*  src/
CMakeCache.txt	pyproject.toml				     test_cpp.py
CMakeLists.txt	requirements.txt
data/		sample_data/


In [30]:
import os
import shutil
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.models import ResNet50_Weights
from torch.utils.data import DataLoader
from tqdm import tqdm
from pathlib import Path

# Hardware check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware: Using {device}\n")

Hardware: Using cuda



## Organizing the Data for the model
- Grabbing the required EyePACS images
- Creating directory structure for normal and glaucoma images
- Sorting the images into their correct buckets based on the dataframes
- Applying image transformations: resizing pictures to 224x224, flipping them to prevent overfitting, and standardizing colors
- Initializing DataLoaders: setting up the pipeline to stream batches of 32 images directly to the GPU

In [31]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import ocular_cpp

def fetch_and_organize_data(train_df, val_df, test_df):
    """
    Download the dataset and sort the images into folders
    based on the dataframes passed into the function.
    """
    eyepacsSrcDir = kagglehub.dataset_download("deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2")
    baseOutputDir = "dataset/split_data"

    # Create physical folders with numbering for alphabetical reading
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(baseOutputDir, split, '0_normal'), exist_ok=True)
        os.makedirs(os.path.join(baseOutputDir, split, '1_glaucoma'), exist_ok=True)

    # Indexing the images
    eyepacsImageMap = {filePath.name: str(filePath) for filePath in Path(eyepacsSrcDir).rglob('*.jpg')}

    def copy_images(df, targetSplit):
        missing = 0
        for index, row in tqdm(df.iterrows(), total=len(df), desc=f"Sorting {targetSplit}"):
            fileName = row['file_name']
            labelDir = '1_glaucoma' if int(row['label_binary']) == 1 else '0_normal'
            destPath = os.path.join(baseOutputDir, targetSplit, labelDir, fileName)
            srcPath = eyepacsImageMap.get(fileName)

            if srcPath and os.path.exists(srcPath):
                shutil.copy(srcPath, destPath)
            else:
                missing += 1
        if missing > 0:
            print(f"Skipped {missing} missing files in {targetSplit}.")

    copy_images(train_df, 'train')
    copy_images(val_df, 'val')
    copy_images(test_df, 'test')

    return baseOutputDir

class OcularCppDataset(Dataset):
    def __init__(self, root_dir, is_train=False):
        self.root_dir = root_dir
        self.is_train = is_train
        self.image_paths = []
        self.labels = []
        self.classes = ['0_normal', '1_glaucoma']
        self.class_to_idx = {'0_normal': 0, '1_glaucoma': 1}

        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            label = self.class_to_idx[class_name]
            if os.path.exists(class_dir):
                for file_name in os.listdir(class_dir):
                    if file_name.endswith('.jpg'):
                        self.image_paths.append(os.path.join(class_dir, file_name))
                        self.labels.append(label)

        # Only random flip for training
        self.random_flip = transforms.RandomHorizontalFlip(p=0.5)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Returns the (224, 224, 3) array from C++
        np_img = ocular_cpp.prepare_input_data(img_path)

        # PERMUTE: Change (Height, Width, Channels) to (Channels, Height, Width)
        # 224, 224, 3 -> 3, 224, 224 opencv structure to pytorch structure
        np_img = np_img.transpose((2, 0, 1))

        img_tensor = torch.from_numpy(np_img)

        # Apply random flipping if in training mode
        if self.is_train:
            img_tensor = self.random_flip(img_tensor)

        return img_tensor, label

def create_dataloaders(baseOutputDir):
    """
    Connects PyTorch to the sorted folders and sets up the GPU streaming using C++.
    """
    trainDataset = OcularCppDataset(os.path.join(baseOutputDir, 'train'), is_train=True)
    valDataset = OcularCppDataset(os.path.join(baseOutputDir, 'val'), is_train=False)

    trainLoader = DataLoader(trainDataset, batch_size=32, shuffle=True, num_workers=2)
    valLoader = DataLoader(valDataset, batch_size=32, shuffle=False, num_workers=2)

    print(f"\nC++ Dataloaders Ready! Class mapping: {trainDataset.class_to_idx}")
    return trainLoader, valLoader, len(trainDataset), len(valDataset)

##Model Configuration
- Downloading ResNet50: pre-trained model
- Maintaining existing image training
- Adjusting image classification buckets: Dogs, Cats, Planes, etc -> only Glaucoma and Normal eye images
- Load the model into the project
- Setting Loss Function : a grading system for the model to be checked against
- Adding a optimizer to adjust the weights to improve the model after each training round

In [32]:
def build_resnet50(device):
    """
    Downloads the pre-trained ResNet50 model, freezes its core layers,
    and swaps the final classification layer to output 2 classes.
    """
    print("Downloading pre-trained ResNet50...")

    # Download the pre-trained model weights
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

    # Freeze the existing layers
    # This stops PyTorch from overwriting the basic trained
    # edge/texture detection it already knows
    for param in model.parameters():
        param.requires_grad = False

    # Swap the final classification layer
    # ResNet50 defaults to 1000 classes. We override it to output
    # 2 classifications (glaucoma and normal)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 2) # 2 classes instead of 1000 (glaucoma, normal)

    # Load model
    model = model.to(device)

    # Define Loss Function and Optimizer for categorization (using cross entropy loss)
    criterion = nn.CrossEntropyLoss()

    # Pass the parameters of the new layer to the optimizer with
    # Adaptive Moment Estimation
    # Can swap Adam with AdamW if overfitting occurs
    optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

    print(f"Final layer successfully swapped: {num_ftrs} inputs -> 2 outputs.")

    return model, criterion, optimizer

##Training the model
- Defining the number of rounds (epochs) the model will study the entire dataset.
- Setting the model into active "learning" mode.
- Streaming the eye images to the GPU in small batches.
- The model predicts, calculates its error against the grading rubric, and the optimizer mathematically adjusts the weights to improve.

In [33]:
def train_model(model, trainLoader, valLoader, train_size, val_size, criterion, optimizer, device, epochs=25):
    """Executes the training and validation. Then, saves the best model state."""
    print(f"Starting Training with {epochs} Epochs...")
    print(f"Feeding {train_size} images per round.\n")

    best_val_loss = float('inf')

    for epoch in range(epochs):
        print(f'Epoch {epoch+1}/{epochs}')
        print('-' * 20)

        #Training the model
        model.train()  # Sets the model to "learning" mode
        running_loss = 0.0
        running_corrects = 0

        # Stream the images through the GPU
        for inputs, labels in trainLoader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Predict, check error, adjust weights
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Directly grab the winning bucket index (0 or 1)
            preds = torch.argmax(outputs, dim=1)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / train_size
        epoch_acc = running_corrects.double() / train_size
        print(f'Train Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

        # Validating the model
        model.eval() # Change the model mode
        val_loss, val_corrects = 0.0, 0

        # Turn off the optimizer so it doesn't try to learn the test answers
        with torch.no_grad():
            for inputs, labels in valLoader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                preds = torch.argmax(outputs, dim=1)

                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)

        val_epoch_loss = val_loss / val_size
        val_epoch_acc = val_corrects.double() / val_size
        print(f'Val Loss:   {val_epoch_loss:.4f} | Acc: {val_epoch_acc:.4f}')

        # Save the best model state
        if val_epoch_loss < best_val_loss:
            best_val_loss = val_epoch_loss
            torch.save(model.state_dict(), 'best_glaucoma_resnet.pth')
            print(f"Saved model at Epoch {epoch+1}\n")

## Implementation
- Filtering the data: isolating only the EyePACS images from the master dataframes
- Preparing the pipeline: fetch, sort, and stream the images into the model
- Set the ResNet50 model parameters
- Start training and validation

In [34]:
print("Filtering Dataframes for EyePACS...")
X_train_eye = X_train[X_train['source_dataset'].str.contains('EyePACS', na=False)]
X_val_eye = X_val[X_val['source_dataset'].str.contains('EyePACS', na=False)]
X_test_eye = X_test[X_test['source_dataset'].str.contains('EyePACS', na=False)]

# Gather the data
data_dir = fetch_and_organize_data(X_train_eye, X_val_eye, X_test_eye)
train_loader, val_loader, train_size, val_size = create_dataloaders(data_dir)

# Build Model
model, criterion, optimizer = build_resnet50(device)

# Train the model
print(f"\nStarting Training Engine for 25 Epochs...")
train_model(model, train_loader, val_loader, train_size, val_size, criterion, optimizer, device, epochs=25)

Filtering Dataframes for EyePACS...
Using Colab cache for faster access to the 'glaucoma-dataset-eyepacs-airogs-light-v2' dataset.


Sorting test: 100%|██████████| 1881/1881 [00:05<00:00, 329.29it/s]



C++ Dataloaders Ready! Class mapping: {'0_normal': 0, '1_glaucoma': 1}
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 159MB/s]


Final layer successfully swapped: 2048 inputs -> 2 outputs.

Starting Training Engine for 25 Epochs...
Starting Training with 25 Epochs...
Feeding 6679 images per round.

Epoch 1/25
--------------------
Train Loss: 0.5831 | Acc: 0.6970
Val Loss:   0.4748 | Acc: 0.7796
Saved model at Epoch 1

Epoch 2/25
--------------------
Train Loss: 0.5169 | Acc: 0.7479
Val Loss:   0.4512 | Acc: 0.8010
Saved model at Epoch 2

Epoch 3/25
--------------------
Train Loss: 0.5015 | Acc: 0.7566
Val Loss:   0.4465 | Acc: 0.7929
Saved model at Epoch 3

Epoch 4/25
--------------------
Train Loss: 0.5006 | Acc: 0.7592
Val Loss:   0.4473 | Acc: 0.7969
Epoch 5/25
--------------------
Train Loss: 0.4941 | Acc: 0.7609
Val Loss:   0.4815 | Acc: 0.7755
Epoch 6/25
--------------------
Train Loss: 0.4954 | Acc: 0.7628
Val Loss:   0.5118 | Acc: 0.7571
Epoch 7/25
--------------------
Train Loss: 0.4902 | Acc: 0.7669
Val Loss:   0.5793 | Acc: 0.7286
Epoch 8/25
--------------------
Train Loss: 0.5046 | Acc: 0.7583
Val Lo